# E1: Episodic Trace Memory for Code Generation

Can an LLM generate better code by retrieving **similar past solved problems** as procedural memory?

### Conditions

| Condition | Description |
|-----------|-------------|
| **NoMemory** | Zero-shot |
| **RandomFewShot** | k random MBPP solutions |
| **FlatCosine** | k most similar by numpy cosine (no CVX) |
| **CVX-Episodic** | k most similar via CVX search, full episode |
| **CVX-Causal** | Match ANY step via CVX, return only the **continuation** (what happened after a similar state) |

### Key Hypothesis: CVX-Causal

Standard retrieval asks: *"Find similar problems."*
Causal retrieval asks: *"Find where someone was in a state like mine, and show me what they did next."*

CVX-Causal searches across ALL steps (problem, plan, solution) — not just problem descriptions.
When it finds a match at step N of episode E, it returns steps N+1..end: the **resolution path from that state**.
This requires temporal structure (episode encoding + trajectory) that FlatCosine cannot provide.

### Experimental Design

1. **Validation** (problems 0–81): sweep top-k ∈ {1, 3, 5, 7}, T=0
2. **Test** (problems 82–163): best k, T=0.2, 5 seeds → confidence intervals
3. **Statistical testing**: McNemar + paired t-test

In [1]:
import os, json, time, hashlib
import numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer
from openai import OpenAI
from scipy import stats

# === MODEL CONFIG ===
USE_OLLAMA = True
OLLAMA_MODEL = 'qwen2.5-coder:7b-instruct'
OLLAMA_URL = 'http://hpc.glaciar.lab:11434/v1'

if USE_OLLAMA:
    client = OpenAI(base_url=OLLAMA_URL, api_key='ollama')
    LLM_MODEL = OLLAMA_MODEL
else:
    assert os.environ.get('OPENAI_API_KEY'), 'Set OPENAI_API_KEY'
    client = OpenAI()
    LLM_MODEL = 'gpt-4o-mini'

EMBED_MODEL = 'all-MiniLM-L6-v2'

# === EXPERIMENTAL DESIGN ===
VAL_SPLIT = 82       # problems 0..81 for validation (top-k sweep)
N_SEEDS = 5           # test seeds for confidence intervals
TEMPERATURE = 0.2     # test temperature (val uses T=0)
K_CANDIDATES = [1, 3, 5, 7]  # top-k values to sweep

DATA_DIR = Path('../data/episodic')
DATA_DIR.mkdir(parents=True, exist_ok=True)

embedder = SentenceTransformer(EMBED_MODEL)
D = embedder.get_sentence_embedding_dimension()
print(f'Embedding: {EMBED_MODEL} (D={D})')
print(f'LLM: {LLM_MODEL} ({"Ollama" if USE_OLLAMA else "OpenAI"})')
print(f'Design: val={VAL_SPLIT} problems, test={164-VAL_SPLIT} problems, seeds={N_SEEDS}, T={TEMPERATURE}')

/opt/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8400.03it/s]


BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding: all-MiniLM-L6-v2 (D=384)
LLM: qwen2.5-coder:7b-instruct (Ollama)
Design: val=82 problems, test=82 problems, seeds=5, T=0.2


## 1. Load Datasets

In [2]:
# Download datasets if not cached
from datasets import load_dataset

# MBPP — training corpus
mbpp = load_dataset('google-research-datasets/mbpp', 'sanitized', split='train+test+prompt')
print(f'MBPP: {len(mbpp)} problems')

# HumanEval — test benchmark
humaneval = load_dataset('openai/openai_humaneval', split='test')
print(f'HumanEval: {len(humaneval)} problems')

# Preview
print(f'\nMBPP example:')
print(f'  Task: {mbpp[0]["prompt"][:100]}...')
print(f'  Code: {mbpp[0]["code"][:100]}...')
print(f'\nHumanEval example:')
print(f'  Prompt: {humaneval[0]["prompt"][:100]}...')

MBPP: 384 problems


HumanEval: 164 problems

MBPP example:
  Task: Write a python function to find the first repeated character in a given string....
  Code: def first_repeated_char(str1):
  for index,c in enumerate(str1):
    if str1[:index+1].count(c) > 1:...

HumanEval example:
  Prompt: from typing import List


def has_close_elements(numbers: List[float], threshold: float) -> bool:
  ...


## 2. Build Episodic Memory from MBPP

Each MBPP problem becomes a 3-step episode in CVX:
- Step 0: problem description (embed the prompt)
- Step 1: solution approach (embed the code's docstring/first comment)
- Step 2: full solution (embed the complete code)

Metadata includes `reward=1.0` (all MBPP solutions pass their tests), `task_id`, and the actual code text.

In [3]:
import chronos_vector as cvx

INDEX_PATH = str(DATA_DIR / 'mbpp_episodic.cvx')
FLAT_EMB_PATH = str(DATA_DIR / 'mbpp_problem_embeddings.npy')

# Episode encoding: timestamp = ep_idx * 1000 + step_index
def encode_entity(episode_id, step_index):
    return (episode_id << 16) | step_index

def decode_entity(entity_id):
    return entity_id >> 16, entity_id & 0xFFFF

# Build or load CVX index
if os.path.exists(INDEX_PATH):
    index = cvx.TemporalIndex.load(INDEX_PATH)
    print(f'Loaded cached CVX index: {len(index)} points')
else:
    print('Building episodic memory from MBPP...')
    index = cvx.TemporalIndex(m=16, ef_construction=100, ef_search=50)
    
    episode_metadata = {}
    
    for ep_idx, problem in enumerate(mbpp):
        prompt = problem['prompt']
        code = problem['code']
        task_id = problem.get('task_id', ep_idx)
        
        # Step 0: problem description
        v_problem = embedder.encode(prompt).tolist()
        eid_0 = encode_entity(ep_idx, 0)
        index.insert(entity_id=eid_0, timestamp=ep_idx * 1000, vector=v_problem)
        
        # Step 1: solution approach (first line of code as plan)
        first_line = code.split('\n')[0] if code else prompt
        v_plan = embedder.encode(f'Plan: {first_line}').tolist()
        eid_1 = encode_entity(ep_idx, 1)
        index.insert(entity_id=eid_1, timestamp=ep_idx * 1000 + 1, vector=v_plan)
        
        # Step 2: full solution
        v_solution = embedder.encode(code[:500]).tolist()
        eid_2 = encode_entity(ep_idx, 2)
        index.insert(entity_id=eid_2, timestamp=ep_idx * 1000 + 2, vector=v_solution)
        
        episode_metadata[ep_idx] = {
            'prompt': prompt,
            'code': code,
            'task_id': task_id,
            'reward': 1.0,
        }
    
    index.save(INDEX_PATH)
    with open(str(DATA_DIR / 'mbpp_metadata.json'), 'w') as f:
        json.dump(episode_metadata, f)
    print(f'Built CVX index: {len(index)} points ({len(mbpp)} episodes × 3 steps)')

# Load metadata
with open(str(DATA_DIR / 'mbpp_metadata.json')) as f:
    episode_metadata = {int(k): v for k, v in json.load(f).items()}

# Build or load flat embedding matrix for cosine baseline
if os.path.exists(FLAT_EMB_PATH):
    mbpp_problem_embeddings = np.load(FLAT_EMB_PATH)
    print(f'Loaded cached flat embeddings: {mbpp_problem_embeddings.shape}')
else:
    print('Building flat embedding matrix (problem descriptions only)...')
    mbpp_problem_embeddings = embedder.encode(
        [episode_metadata[i]['prompt'] for i in range(len(episode_metadata))]
    )
    np.save(FLAT_EMB_PATH, mbpp_problem_embeddings)
    print(f'Built flat embeddings: {mbpp_problem_embeddings.shape}')

print(f'{len(episode_metadata)} episodes with metadata, flat matrix {mbpp_problem_embeddings.shape}')

Loaded cached CVX index: 1152 points
Loaded cached flat embeddings: (384, 384)
384 episodes with metadata, flat matrix (384, 384)


## 3. Retrieval Functions

In [4]:
def retrieve_episodes_cvx(query_text, top_k=3):
    """Retrieve similar MBPP episodes via CVX temporal search.
    Only matches step_0 (problem descriptions). Returns full episodes.
    """
    query_vec = embedder.encode(query_text).tolist()
    results = index.search(vector=query_vec, k=top_k * 10)
    
    seen_episodes = set()
    episodes = []
    for node_id, timestamp, score in results:
        ep_idx = timestamp // 1000
        step_idx = timestamp % 1000
        
        if step_idx != 0:
            continue
        if ep_idx in seen_episodes:
            continue
        seen_episodes.add(ep_idx)
        
        if ep_idx in episode_metadata:
            episodes.append({
                'episode_id': ep_idx,
                'similarity': score,
                **episode_metadata[ep_idx],
            })
        
        if len(episodes) >= top_k:
            break
    
    return episodes


def retrieve_causal_cvx(query_text, top_k=3):
    """CVX-Causal: search across ALL steps, return the CONTINUATION from match point.
    
    This is fundamentally different from flat/episodic retrieval:
    - Searches problem, plan, AND solution embeddings (not just problems)
    - Returns only what happened AFTER the matched state
    - If matched at step 0 (problem): returns plan + solution
    - If matched at step 1 (plan): returns only the solution
    - If matched at step 2 (solution): returns the solution itself
    
    FlatCosine cannot do this — it has no temporal structure or step ordering.
    """
    query_vec = embedder.encode(query_text).tolist()
    results = index.search(vector=query_vec, k=top_k * 15)
    
    seen_episodes = set()
    episodes = []
    for node_id, timestamp, score in results:
        ep_idx = timestamp // 1000
        match_step = timestamp % 1000
        
        if ep_idx in seen_episodes:
            continue
        seen_episodes.add(ep_idx)
        
        if ep_idx in episode_metadata:
            episodes.append({
                'episode_id': ep_idx,
                'match_step': match_step,  # which step was matched
                'similarity': score,
                **episode_metadata[ep_idx],
            })
        
        if len(episodes) >= top_k:
            break
    
    return episodes


def retrieve_flat_cosine(query_text, top_k=3):
    """Flat numpy cosine on problem description embeddings only."""
    query_vec = embedder.encode(query_text)
    query_norm = query_vec / (np.linalg.norm(query_vec) + 1e-8)
    emb_norms = mbpp_problem_embeddings / (np.linalg.norm(mbpp_problem_embeddings, axis=1, keepdims=True) + 1e-8)
    sims = emb_norms @ query_norm
    
    top_indices = np.argsort(sims)[::-1][:top_k]
    episodes = []
    for idx in top_indices:
        idx = int(idx)
        if idx in episode_metadata:
            episodes.append({
                'episode_id': idx,
                'similarity': float(sims[idx]),
                **episode_metadata[idx],
            })
    return episodes


def retrieve_random(top_k=3):
    """Random baseline."""
    indices = np.random.choice(len(episode_metadata), size=top_k, replace=False)
    return [episode_metadata[int(i)] for i in indices]


def format_episodes_for_prompt(episodes):
    """Standard format: full problem + solution."""
    lines = []
    for i, ep in enumerate(episodes, 1):
        lines.append(f'### Example {i} (from similar past problem)')
        lines.append(f'Problem: {ep["prompt"]}')
        lines.append(f'Solution:')
        lines.append(f'```python')
        lines.append(ep['code'])
        lines.append(f'```')
        lines.append('')
    return '\n'.join(lines)


def format_causal_for_prompt(episodes):
    """Causal format: show only the CONTINUATION from the matched state.
    
    - Matched at step 0 (problem): show approach + solution
    - Matched at step 1 (plan/approach): show only the solution
    - Matched at step 2 (solution): show the solution as a reference
    """
    lines = []
    for i, ep in enumerate(episodes, 1):
        match_step = ep.get('match_step', 0)
        code = ep['code']
        first_line = code.split('\n')[0] if code else ''
        
        if match_step == 0:
            # Matched on problem description → show approach + solution
            lines.append(f'### Example {i} (similar problem → here is how it was solved)')
            lines.append(f'Problem: {ep["prompt"]}')
            lines.append(f'Approach: {first_line}')
            lines.append(f'Solution:')
            lines.append(f'```python')
            lines.append(code)
            lines.append(f'```')
        elif match_step == 1:
            # Matched on plan/approach → show only the solution
            lines.append(f'### Example {i} (similar approach detected → here is the resulting code)')
            lines.append(f'The approach "{first_line}" led to:')
            lines.append(f'```python')
            lines.append(code)
            lines.append(f'```')
        else:
            # Matched on solution code → show as reference
            lines.append(f'### Example {i} (similar code pattern found)')
            lines.append(f'This code solved: {ep["prompt"][:100]}')
            lines.append(f'```python')
            lines.append(code)
            lines.append(f'```')
        lines.append('')
    return '\n'.join(lines)


# Test all retrieval methods
test_query = humaneval[0]['prompt']
print(f'Query: {test_query[:80]}...\n')

for name, fn in [('CVX-Episodic', retrieve_episodes_cvx),
                 ('FlatCosine', retrieve_flat_cosine),
                 ('CVX-Causal', retrieve_causal_cvx)]:
    eps = fn(test_query, top_k=3)
    print(f'{name}: {len(eps)} episodes')
    for ep in eps:
        step_info = f' (matched step {ep["match_step"]})' if 'match_step' in ep else ''
        print(f'  [{ep["episode_id"]}] sim={ep["similarity"]:.3f}{step_info}: {ep["prompt"][:50]}...')
    print()

Query: from typing import List


def has_close_elements(numbers: List[float], threshold...

CVX-Episodic: 3 episodes
  [82] sim=0.909: Write a python function to get the difference betw...
  [133] sim=0.916: Write a python function to find smallest number in...
  [77] sim=0.934: Write a python function to find the minimum differ...



FlatCosine: 3 episodes
  [82] sim=0.545: Write a python function to get the difference betw...
  [133] sim=0.542: Write a python function to find smallest number in...
  [77] sim=0.533: Write a python function to find the minimum differ...

CVX-Causal: 3 episodes
  [82] sim=0.909 (matched step 0): Write a python function to get the difference betw...
  [133] sim=0.916 (matched step 0): Write a python function to find smallest number in...
  [77] sim=0.934 (matched step 0): Write a python function to find the minimum differ...



## 4. LLM Code Generation

In [5]:
def generate_code(problem_prompt, examples_text='', model=LLM_MODEL, temperature=0.0, seed=None):
    """Generate Python code for a HumanEval problem."""
    system = (
        'You are an expert Python programmer. Complete the function based on '
        'the docstring. Return ONLY the function body (no explanation, no markdown). '
        'The function signature is already provided.'
    )
    
    user_parts = []
    if examples_text:
        user_parts.append('Here are similar solved problems for reference:\n')
        user_parts.append(examples_text)
        user_parts.append('\n---\nNow solve this problem:\n')
    user_parts.append(problem_prompt)
    
    kwargs = dict(
        model=model,
        messages=[
            {'role': 'system', 'content': system},
            {'role': 'user', 'content': '\n'.join(user_parts)},
        ],
        temperature=temperature,
        max_tokens=1024,
    )
    if seed is not None:
        kwargs['seed'] = seed
    
    response = client.chat.completions.create(**kwargs)
    return response.choices[0].message.content


def extract_function_body(generated, prompt):
    """Extract the function body from generated code."""
    code = generated.strip()
    if code.startswith('```'):
        code = code.split('\n', 1)[1] if '\n' in code else code[3:]
    if code.endswith('```'):
        code = code[:-3]
    code = code.strip()
    return code


def evaluate_solution(prompt, generated_body, test_code, entry_point):
    """Run the test cases against the generated solution."""
    full_code = prompt + generated_body + '\n' + test_code
    try:
        exec_globals = {}
        exec(full_code, exec_globals)
        check_fn = f'check({entry_point})'
        exec(check_fn, exec_globals)
        return True
    except Exception:
        return False


def run_evaluation(problems, conditions, temperature=0.0, seed=None, top_k=3):
    """Run all conditions on a set of problems. Returns {cond_name: [{'task_id', 'passed'}]}."""
    results = {name: [] for name in conditions}
    for i, problem in enumerate(problems):
        prompt = problem['prompt']
        test_code = problem['test']
        entry_point = problem['entry_point']
        task_id = problem['task_id']
        
        for cond_name, get_context in conditions.items():
            context = get_context(prompt, top_k)
            generated = generate_code(prompt, context, temperature=temperature, seed=seed)
            body = extract_function_body(generated, prompt)
            passed = evaluate_solution(prompt, body, test_code, entry_point)
            results[cond_name].append({
                'task_id': task_id,
                'passed': passed,
            })
        
        if (i + 1) % 20 == 0:
            rates = {name: sum(r['passed'] for r in res) / len(res)
                     for name, res in results.items()}
            print(f'  [{i+1}/{len(problems)}] {rates}')
    
    return results


# Quick test
test_gen = generate_code(humaneval[0]['prompt'])
print(f'Generated ({len(test_gen)} chars):\n{test_gen[:200]}...')

Generated (266 chars):
```python
def has_close_elements(numbers: List[float], threshold: float) -> bool:
    for i in range(len(numbers)):
        for j in range(i + 1, len(numbers)):
            if abs(numbers[i] - numbers...


## 5. Validation: Top-k Sweep

Use the first 82 HumanEval problems to find the optimal k for retrieval-based conditions.
Single pass with T=0 (deterministic) for speed.

In [6]:
# Condition factories — accept (prompt, top_k) for uniformity
conditions = {
    'NoMemory':      lambda p, k: '',
    'RandomFewShot': lambda p, k: format_episodes_for_prompt(retrieve_random(top_k=k)),
    'FlatCosine':    lambda p, k: format_episodes_for_prompt(retrieve_flat_cosine(p, top_k=k)),
    'CVX-Episodic':  lambda p, k: format_episodes_for_prompt(retrieve_episodes_cvx(p, top_k=k)),
    'CVX-Causal':    lambda p, k: format_causal_for_prompt(retrieve_causal_cvx(p, top_k=k)),
}

val_problems = list(humaneval)[:VAL_SPLIT]
print(f'=== VALIDATION SWEEP (n={len(val_problems)}, T=0) ===\n')

val_sweep = {}

for k in K_CANDIDATES:
    print(f'--- k={k} ---')
    results = run_evaluation(val_problems, conditions, temperature=0.0, top_k=k)
    val_sweep[k] = {}
    for cond, res in results.items():
        rate = sum(r['passed'] for r in res) / len(res)
        val_sweep[k][cond] = rate
    print(f'  Results: { {c: f"{r:.1%}" for c, r in val_sweep[k].items()} }\n')

print('=== VALIDATION SUMMARY ===')
header = ' | '.join(f'{c:>14}' for c in conditions)
print(f'{"k":>3} | {header}')
print('-' * (4 + 17 * len(conditions)))
for k in K_CANDIDATES:
    row = ' | '.join(f'{val_sweep[k][c]:>14.1%}' for c in conditions)
    print(f'{k:>3} | {row}')

# Select best k by CVX-Causal (the new condition to test)
best_k = max(K_CANDIDATES, key=lambda k: val_sweep[k]['CVX-Causal'])
print(f'\nBest k (by CVX-Causal on val): k={best_k} ({val_sweep[best_k]["CVX-Causal"]:.1%})')

=== VALIDATION SWEEP (n=82, T=0) ===

--- k=1 ---


  [20/82] {'NoMemory': 0.6, 'RandomFewShot': 0.85, 'FlatCosine': 0.85, 'CVX-Episodic': 0.85, 'CVX-Causal': 0.75}


  [40/82] {'NoMemory': 0.55, 'RandomFewShot': 0.825, 'FlatCosine': 0.875, 'CVX-Episodic': 0.85, 'CVX-Causal': 0.775}


  [60/82] {'NoMemory': 0.5333333333333333, 'RandomFewShot': 0.8333333333333334, 'FlatCosine': 0.85, 'CVX-Episodic': 0.8333333333333334, 'CVX-Causal': 0.7666666666666667}


  [80/82] {'NoMemory': 0.625, 'RandomFewShot': 0.8375, 'FlatCosine': 0.85, 'CVX-Episodic': 0.8375, 'CVX-Causal': 0.8125}


...

  [20/82] {'NoMemory': 0.6, 'RandomFewShot': 0.9, 'FlatCosine': 0.9, 'CVX-Episodic': 0.9, 'CVX-Causal': 0.85}


  [40/82] {'NoMemory': 0.55, 'RandomFewShot': 0.825, 'FlatCosine': 0.85, 'CVX-Episodic': 0.85, 'CVX-Causal': 0.8}


  [60/82] {'NoMemory': 0.5333333333333333, 'RandomFewShot': 0.8166666666666667, 'FlatCosine': 0.8166666666666667, 'CVX-Episodic': 0.8166666666666667, 'CVX-Causal': 0.8166666666666667}


  [80/82] {'NoMemory': 0.625, 'RandomFewShot': 0.825, 'FlatCosine': 0.825, 'CVX-Episodic': 0.825, 'CVX-Causal': 0.825}


  Results: {'NoMemory': '63.4%', 'RandomFewShot': '82.9%', 'FlatCosine': '82.9%', 'CVX-Episodic': '82.9%', 'CVX-Causal': '82.9%'}

=== VALIDATION SUMMARY ===
  k |       NoMemory |  RandomFewShot |     FlatCosine |   CVX-Episodic |     CVX-Causal
-----------------------------------------------------------------------------------------
  1 |          63.4% |          84.1% |          85.4% |          84.1% |          81.7%
  3 |          63.4% |          78.0% |          86.6% |          86.6% |          80.5%
  5 |          63.4% |          82.9% |          84.1% |          85.4% |          85.4%
  7 |          63.4% |          82.9% |          82.9% |          82.9% |          82.9%

Best k (by CVX-Causal on val): k=5 (85.4%)


In [7]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

colors = {
    'NoMemory': '#95a5a6', 'RandomFewShot': '#3498db', 'FlatCosine': '#f39c12',
    'CVX-Episodic': '#e74c3c', 'CVX-Causal': '#2ecc71',
}

# Validation sweep
fig = go.Figure()
for cond in [c for c in conditions if c != 'NoMemory']:
    rates = [val_sweep[k][cond] * 100 for k in K_CANDIDATES]
    fig.add_trace(go.Scatter(
        x=K_CANDIDATES, y=rates,
        mode='lines+markers', name=cond,
        marker=dict(size=8), line=dict(color=colors.get(cond)),
    ))

nm_rate = val_sweep[K_CANDIDATES[0]]['NoMemory'] * 100
fig.add_hline(y=nm_rate, line_dash='dash', line_color='gray',
              annotation_text=f'NoMemory ({nm_rate:.0f}%)')

fig.update_layout(
    title=f'Validation: pass@1 vs top-k (n={len(val_problems)}, T=0)',
    xaxis_title='top-k', yaxis_title='pass@1 (%)',
    yaxis=dict(range=[0, 100]), height=400,
)
fig.show()

## 6. Test Evaluation (Multiple Seeds)

Evaluate on the held-out test split (problems 82–163) using the best k from validation.
T=0.2 with 5 different seeds → mean ± std for confidence intervals.

In [8]:
test_problems = list(humaneval)[VAL_SPLIT:]
print(f'=== TEST EVALUATION (n={len(test_problems)}, k={best_k}, T={TEMPERATURE}, seeds={N_SEEDS}) ===\n')

# Run N_SEEDS times
all_seed_results = {}  # {seed: {cond: [{'task_id', 'passed'}]}}

for seed_idx in range(N_SEEDS):
    seed = 42 + seed_idx
    print(f'--- Seed {seed} ({seed_idx+1}/{N_SEEDS}) ---')
    np.random.seed(seed)  # for RandomFewShot reproducibility per seed
    results = run_evaluation(test_problems, conditions, temperature=TEMPERATURE, seed=seed, top_k=best_k)
    all_seed_results[seed] = results
    
    rates = {c: sum(r['passed'] for r in res) / len(res) for c, res in results.items()}
    print(f'  pass@1: { {c: f"{r:.1%}" for c, r in rates.items()} }\n')

# Aggregate across seeds
import pandas as pd

test_summary = []
for cond in conditions:
    seed_rates = []
    for seed, results in all_seed_results.items():
        rate = sum(r['passed'] for r in results[cond]) / len(results[cond])
        seed_rates.append(rate)
    test_summary.append({
        'condition': cond,
        'mean': np.mean(seed_rates),
        'std': np.std(seed_rates),
        'min': np.min(seed_rates),
        'max': np.max(seed_rates),
        'seeds': seed_rates,
    })

df_test = pd.DataFrame(test_summary)

print('=== TEST RESULTS (mean ± std across seeds) ===')
for _, row in df_test.iterrows():
    print(f'  {row["condition"]:15s}: {row["mean"]:.1%} ± {row["std"]:.1%}  (range: {row["min"]:.1%}–{row["max"]:.1%})')

# Also aggregate per-problem pass rates (across seeds) for McNemar
# For each problem, a condition "passes" if it passes in majority of seeds
problem_pass = {cond: [] for cond in conditions}
for prob_idx in range(len(test_problems)):
    for cond in conditions:
        n_pass = sum(all_seed_results[s][cond][prob_idx]['passed'] for s in all_seed_results)
        problem_pass[cond].append(n_pass / N_SEEDS)  # fraction of seeds that passed

=== TEST EVALUATION (n=82, k=5, T=0.2, seeds=5) ===

--- Seed 42 (1/5) ---


  [20/82] {'NoMemory': 0.7, 'RandomFewShot': 0.75, 'FlatCosine': 0.75, 'CVX-Episodic': 0.75, 'CVX-Causal': 0.75}


  [40/82] {'NoMemory': 0.75, 'RandomFewShot': 0.75, 'FlatCosine': 0.825, 'CVX-Episodic': 0.825, 'CVX-Causal': 0.8}


<string>:17: SyntaxWarning: invalid escape sequence '\S'


  [60/82] {'NoMemory': 0.75, 'RandomFewShot': 0.7, 'FlatCosine': 0.75, 'CVX-Episodic': 0.75, 'CVX-Causal': 0.7333333333333333}


...

  [40/82] {'NoMemory': 0.725, 'RandomFewShot': 0.775, 'FlatCosine': 0.825, 'CVX-Episodic': 0.825, 'CVX-Causal': 0.825}


<string>:18: SyntaxWarning: invalid escape sequence '\s'


  [60/82] {'NoMemory': 0.7166666666666667, 'RandomFewShot': 0.7333333333333333, 'FlatCosine': 0.75, 'CVX-Episodic': 0.75, 'CVX-Causal': 0.75}


  [80/82] {'NoMemory': 0.7625, 'RandomFewShot': 0.75, 'FlatCosine': 0.75, 'CVX-Episodic': 0.7625, 'CVX-Causal': 0.75}


  pass@1: {'NoMemory': '75.6%', 'RandomFewShot': '74.4%', 'FlatCosine': '74.4%', 'CVX-Episodic': '75.6%', 'CVX-Causal': '74.4%'}

=== TEST RESULTS (mean ± std across seeds) ===
  NoMemory       : 72.9% ± 2.1%  (range: 69.5%–75.6%)
  RandomFewShot  : 76.6% ± 2.9%  (range: 72.0%–79.3%)
  FlatCosine     : 77.6% ± 2.0%  (range: 74.4%–80.5%)
  CVX-Episodic   : 77.8% ± 1.6%  (range: 75.6%–80.5%)
  CVX-Causal     : 75.1% ± 1.8%  (range: 72.0%–76.8%)


## 7. Results Visualization

In [9]:
# Bar chart with error bars
colors = {'NoMemory': '#95a5a6', 'RandomFewShot': '#3498db', 'FlatCosine': '#f39c12', 'CVX-Episodic': '#e74c3c'}

fig = go.Figure()
for _, row in df_test.iterrows():
    fig.add_trace(go.Bar(
        x=[row['condition']], y=[row['mean'] * 100],
        error_y=dict(type='data', array=[row['std'] * 100], visible=True),
        name=row['condition'],
        marker_color=colors.get(row['condition'], '#333'),
        text=f'{row["mean"]:.1%} ± {row["std"]:.1%}',
        textposition='outside',
    ))

fig.update_layout(
    title=f'Test pass@1: k={best_k}, T={TEMPERATURE}, {N_SEEDS} seeds (n={len(test_problems)})',
    yaxis_title='pass@1 (%)', yaxis=dict(range=[0, 100]),
    height=450, showlegend=False,
)
fig.show()

# Per-seed line plot
fig2 = go.Figure()
for _, row in df_test.iterrows():
    fig2.add_trace(go.Scatter(
        x=list(range(1, N_SEEDS + 1)), y=[s * 100 for s in row['seeds']],
        mode='lines+markers', name=row['condition'],
        line=dict(color=colors.get(row['condition'], '#333')),
    ))

fig2.update_layout(
    title=f'pass@1 by Seed (k={best_k}, T={TEMPERATURE})',
    xaxis_title='Seed', yaxis_title='pass@1 (%)',
    yaxis=dict(range=[0, 100]), height=350,
)
fig2.show()

## 8. Statistical Tests & Ablation Analysis

In [10]:
# === McNemar's Test ===
majority_pass = {cond: np.array([p >= 0.5 for p in problem_pass[cond]]) for cond in conditions}

print('=== McNemar\'s Test (majority-vote, pairwise) ===\n')

pairs = [
    ('CVX-Causal', 'NoMemory'),
    ('CVX-Causal', 'FlatCosine'),
    ('CVX-Causal', 'CVX-Episodic'),
    ('CVX-Causal', 'RandomFewShot'),
    ('CVX-Episodic', 'FlatCosine'),
    ('FlatCosine', 'NoMemory'),
]

for a, b in pairs:
    a_pass, b_pass = majority_pass[a], majority_pass[b]
    n_10 = np.sum(a_pass & ~b_pass)
    n_01 = np.sum(~a_pass & b_pass)
    n_11 = np.sum(a_pass & b_pass)
    n_00 = np.sum(~a_pass & ~b_pass)
    
    if n_10 + n_01 > 0:
        chi2 = (abs(n_10 - n_01) - 1) ** 2 / (n_10 + n_01)
        p_val = 1 - stats.chi2.cdf(chi2, df=1)
    else:
        chi2, p_val = 0, 1.0
    
    sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'ns'
    print(f'{a} vs {b}: {a} only={n_10}, {b} only={n_01} → Net {n_10-n_01:+d}, χ²={chi2:.2f}, p={p_val:.4f} {sig}')

# === Paired t-test on seed-level pass rates ===
print('\n=== Paired t-test (seed-level) ===\n')
for a, b in [('CVX-Causal', 'FlatCosine'), ('CVX-Causal', 'CVX-Episodic'),
             ('CVX-Causal', 'RandomFewShot'), ('CVX-Causal', 'NoMemory')]:
    a_rates = df_test[df_test['condition'] == a]['seeds'].values[0]
    b_rates = df_test[df_test['condition'] == b]['seeds'].values[0]
    t_stat, p_val = stats.ttest_rel(a_rates, b_rates)
    sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'ns'
    diff = np.mean(a_rates) - np.mean(b_rates)
    print(f'{a} vs {b}: Δ={diff:+.1%}, t={t_stat:.3f}, p={p_val:.4f} {sig}')

# === CVX-Causal step distribution ===
print('\n=== CVX-Causal: Which steps get matched? ===')
step_counts = {0: 0, 1: 0, 2: 0}
for problem in test_problems:
    eps = retrieve_causal_cvx(problem['prompt'], top_k=best_k)
    for ep in eps:
        step_counts[ep.get('match_step', 0)] += 1
total = sum(step_counts.values())
for step, count in step_counts.items():
    step_names = {0: 'problem description', 1: 'plan/approach', 2: 'solution code'}
    print(f'  Step {step} ({step_names[step]}): {count}/{total} = {count/max(total,1):.1%}')

# === Retrieval overlap ===
print('\n=== Retrieval Overlap ===')
for a_name, a_fn, b_name, b_fn in [
    ('CVX-Causal', retrieve_causal_cvx, 'CVX-Episodic', retrieve_episodes_cvx),
    ('CVX-Causal', retrieve_causal_cvx, 'FlatCosine', retrieve_flat_cosine),
    ('CVX-Episodic', retrieve_episodes_cvx, 'FlatCosine', retrieve_flat_cosine),
]:
    overlaps = []
    for problem in test_problems:
        a_eps = a_fn(problem['prompt'], top_k=best_k)
        b_eps = b_fn(problem['prompt'], top_k=best_k)
        a_ids = {ep['episode_id'] for ep in a_eps}
        b_ids = {ep['episode_id'] for ep in b_eps}
        overlaps.append(len(a_ids & b_ids) / best_k if best_k > 0 else 0)
    print(f'{a_name} vs {b_name}: {np.mean(overlaps):.1%} overlap')

=== McNemar's Test (majority-vote, pairwise) ===

CVX-Causal vs NoMemory: CVX-Causal only=7, NoMemory only=5 → Net +2, χ²=0.08, p=0.7728 ns
CVX-Causal vs FlatCosine: CVX-Causal only=4, FlatCosine only=4 → Net +0, χ²=0.12, p=0.7237 ns
CVX-Causal vs CVX-Episodic: CVX-Causal only=4, CVX-Episodic only=4 → Net +0, χ²=0.12, p=0.7237 ns
CVX-Causal vs RandomFewShot: CVX-Causal only=6, RandomFewShot only=7 → Net -1, χ²=0.00, p=1.0000 ns
CVX-Episodic vs FlatCosine: CVX-Episodic only=0, FlatCosine only=0 → Net +0, χ²=0.00, p=1.0000 ns
FlatCosine vs NoMemory: FlatCosine only=9, NoMemory only=7 → Net +2, χ²=0.06, p=0.8026 ns

=== Paired t-test (seed-level) ===

CVX-Causal vs FlatCosine: Δ=-2.4%, t=-2.390, p=0.0751 ns
CVX-Causal vs CVX-Episodic: Δ=-2.7%, t=-2.994, p=0.0402 *
CVX-Causal vs RandomFewShot: Δ=-1.5%, t=-0.739, p=0.5012 ns
CVX-Causal vs NoMemory: Δ=+2.2%, t=1.686, p=0.1671 ns

=== CVX-Causal: Which steps get matched? ===


  Step 0 (problem description): 158/410 = 38.5%
  Step 1 (plan/approach): 35/410 = 8.5%
  Step 2 (solution code): 217/410 = 52.9%

=== Retrieval Overlap ===


CVX-Causal vs CVX-Episodic: 61.0% overlap


CVX-Causal vs FlatCosine: 61.0% overlap


CVX-Episodic vs FlatCosine: 97.6% overlap


## 9. Summary

In [11]:
print('=== E1: EPISODIC CODING BENCHMARK — FINAL REPORT ===')
print(f'Model: {LLM_MODEL}')
print(f'Training corpus: MBPP ({len(episode_metadata)} episodes)')
print(f'Embedding: {EMBED_MODEL} (D={D})')
print(f'Validation: HumanEval[0:{VAL_SPLIT}] (n={VAL_SPLIT}), T=0, k sweep: {K_CANDIDATES}')
print(f'Test: HumanEval[{VAL_SPLIT}:164] (n={len(test_problems)}), T={TEMPERATURE}, {N_SEEDS} seeds')
print(f'Best k (from validation): {best_k}')
print()

print('--- Validation (T=0, single pass) ---')
for k in K_CANDIDATES:
    row = '  '.join(f'{c}={val_sweep[k][c]:.1%}' for c in conditions)
    marker = ' <-' if k == best_k else ''
    print(f'  k={k}: {row}{marker}')

print()
print('--- Test (mean +/- std across seeds) ---')
for _, row in df_test.iterrows():
    print(f'  {row["condition"]:15s}: {row["mean"]:.1%} +/- {row["std"]:.1%}')

print()
print('--- Key Findings ---')
cvx_mean = df_test[df_test['condition'] == 'CVX-Episodic']['mean'].values[0]
flat_mean = df_test[df_test['condition'] == 'FlatCosine']['mean'].values[0]
rand_mean = df_test[df_test['condition'] == 'RandomFewShot']['mean'].values[0]
nm_mean = df_test[df_test['condition'] == 'NoMemory']['mean'].values[0]
print(f'  CVX vs NoMemory:      +{cvx_mean - nm_mean:.1%}')
print(f'  CVX vs RandomFewShot: +{cvx_mean - rand_mean:.1%}')
print(f'  CVX vs FlatCosine:    +{cvx_mean - flat_mean:.1%}')
print('  (Retrieval overlap computed in stats section above)')
print()
print('CVX features used: TemporalIndex, insert, search, save/load, episode encoding')

=== E1: EPISODIC CODING BENCHMARK — FINAL REPORT ===
Model: qwen2.5-coder:7b-instruct
Training corpus: MBPP (384 episodes)
Embedding: all-MiniLM-L6-v2 (D=384)
Validation: HumanEval[0:82] (n=82), T=0, k sweep: [1, 3, 5, 7]
Test: HumanEval[82:164] (n=82), T=0.2, 5 seeds
Best k (from validation): 5

--- Validation (T=0, single pass) ---
  k=1: NoMemory=63.4%  RandomFewShot=84.1%  FlatCosine=85.4%  CVX-Episodic=84.1%  CVX-Causal=81.7%
  k=3: NoMemory=63.4%  RandomFewShot=78.0%  FlatCosine=86.6%  CVX-Episodic=86.6%  CVX-Causal=80.5%
  k=5: NoMemory=63.4%  RandomFewShot=82.9%  FlatCosine=84.1%  CVX-Episodic=85.4%  CVX-Causal=85.4% <-
  k=7: NoMemory=63.4%  RandomFewShot=82.9%  FlatCosine=82.9%  CVX-Episodic=82.9%  CVX-Causal=82.9%

--- Test (mean +/- std across seeds) ---
  NoMemory       : 72.9% +/- 2.1%
  RandomFewShot  : 76.6% +/- 2.9%
  FlatCosine     : 77.6% +/- 2.0%
  CVX-Episodic   : 77.8% +/- 1.6%
  CVX-Causal     : 75.1% +/- 1.8%

--- Key Findings ---
  CVX vs NoMemory:      +4.9%
 